# TripMe Part 2 — Raw Data Audit
Attach the `tripme-part02-data` Kaggle Dataset and run all cells. A GPU is not required.

In [ ]:
from pathlib import Path
import shutil

inputs = list(Path('/kaggle/input').rglob('scripts/00_audit_raw.py'))
if not inputs:
    raise FileNotFoundError('Attach the tripme-part02-data Kaggle Dataset')
source_root = inputs[0].parent.parent
work_root = Path('/kaggle/working/tripme-part02-work')
if work_root.exists():
    shutil.rmtree(work_root)
shutil.copytree(source_root / 'data' / 'raw', work_root / 'data' / 'raw')
(work_root / 'scripts').mkdir(parents=True)
shutil.copy2(source_root / 'scripts' / '00_audit_raw.py', work_root / 'scripts' / '00_audit_raw.py')
print('Input:', source_root)
print('Raw files:', len(list((work_root / 'data' / 'raw').rglob('*.jsonl'))))


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(work_root / 'scripts' / '00_audit_raw.py')], cwd=work_root, check=True)


In [ ]:
import csv
import json
from datetime import datetime, timezone

output_dir = Path('/kaggle/working/part02_output')
output_dir.mkdir(parents=True, exist_ok=True)
report = json.loads((work_root / 'data' / 'processed' / 'raw_audit.json').read_text(encoding='utf-8'))
(output_dir / 'raw_audit.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

queue = []
for item in report.get('invalid_or_unverified_coordinates', []):
    queue.append({'issue': 'coordinate_review', **item})
for item in report.get('mojibake_records', []):
    queue.append({'issue': 'encoding_review', **item})
for item in report.get('parse_errors', []):
    queue.append({'issue': 'parse_error', **item})
fields = sorted({key for row in queue for key in row}) or ['issue']
with (output_dir / 'manual_review_queue.csv').open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(queue)

summary = report['summary']
summary_md = '# Part 2 Raw Data Audit Summary\n\n' + '\n'.join(f'- **{k}**: {v}' for k, v in summary.items())
summary_md += f'\n\nManual review queue rows: **{len(queue)}**\n'
(output_dir / 'part02_summary.md').write_text(summary_md, encoding='utf-8')
manifest = {
    'part': 2, 'status': 'audit_complete',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'raw_files': summary.get('files'), 'records': summary.get('records'),
    'manual_review_rows': len(queue),
}
(output_dir / 'part02_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(summary_md)


In [ ]:
archive = shutil.make_archive('/kaggle/working/tripme_part02_output', 'zip', output_dir)
print('Download this file from Notebook Output:')
print(archive)
